In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

('shakespeare.txt', <http.client.HTTPMessage at 0x7be3cf152d80>)

In [4]:
with open("shakespeare.txt", "r") as f:
    text = f.read().lower()

print(len(text))        # how many characters
print(text[:500])       # preview first 500 chars

1115394
first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us kill him, and we'll have corn at our own price.
is't a verdict?

all:
no more talking on't; let it be done: away, away!

second citizen:
one word, good citizens.

first citizen:
we are accounted poor


In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
tokenizer = Tokenizer()

In [7]:
tokenizer.fit_on_texts([text])

In [8]:
len(tokenizer.word_index)

12632

In [9]:
input_sequences = []
for sentence in text.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])

In [10]:
input_sequences

[[88, 269],
 [139, 35],
 [139, 35, 969],
 [139, 35, 969, 143],
 [139, 35, 969, 143, 668],
 [139, 35, 969, 143, 668, 127],
 [139, 35, 969, 143, 668, 127, 15],
 [139, 35, 969, 143, 668, 127, 15, 102],
 [102, 102],
 [88, 269],
 [6, 40],
 [6, 40, 33],
 [6, 40, 33, 1268],
 [6, 40, 33, 1268, 350],
 [6, 40, 33, 1268, 350, 3],
 [6, 40, 33, 1268, 350, 3, 199],
 [6, 40, 33, 1268, 350, 3, 199, 63],
 [6, 40, 33, 1268, 350, 3, 199, 63, 3],
 [6, 40, 33, 1268, 350, 3, 199, 63, 3, 3332],
 [1268, 1268],
 [88, 269],
 [88, 6],
 [88, 6, 91],
 [88, 6, 91, 1141],
 [88, 6, 91, 1141, 231],
 [88, 6, 91, 1141, 231, 11],
 [88, 6, 91, 1141, 231, 11, 2274],
 [88, 6, 91, 1141, 231, 11, 2274, 580],
 [88, 6, 91, 1141, 231, 11, 2274, 580, 3],
 [88, 6, 91, 1141, 231, 11, 2274, 580, 3, 1],
 [88, 6, 91, 1141, 231, 11, 2274, 580, 3, 1, 305],
 [35, 2555],
 [35, 2555, 35],
 [35, 2555, 35, 2555],
 [88, 269],
 [70, 78],
 [70, 78, 502],
 [70, 78, 502, 26],
 [70, 78, 502, 26, 2],
 [70, 78, 502, 26, 2, 369],
 [70, 78, 502, 26, 2

In [11]:
max_len = max([len(x) for x in input_sequences])

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [13]:
padded_input_sequences = pad_sequences(input_sequences, maxlen= max_len, padding='pre')

In [14]:
padded_input_sequences

array([[   0,    0,    0, ...,    0,   88,  269],
       [   0,    0,    0, ...,    0,  139,   35],
       [   0,    0,    0, ...,  139,   35,  969],
       ...,
       [   0,    0,    0, ...,    0, 1062,   25],
       [   0,    0,    0, ..., 1062,   25,  131],
       [   0,    0,    0, ...,   25,  131, 1828]], dtype=int32)

In [15]:
X = padded_input_sequences[:, :-1]

In [16]:
y = padded_input_sequences[:,-1]

In [17]:
y

array([ 269,   35,  969, ...,   25,  131, 1828], dtype=int32)

12632

In [18]:
from tensorflow.keras.utils import to_categorical

In [19]:
y = to_categorical(y, num_classes = 12633)

In [20]:
X.shape

(171312, 15)

In [21]:
y.shape

(171312, 12633)

In [22]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Embedding, Dense, LSTM, Dropout

In [23]:
model = Sequential()
model.add(Embedding(12633, 150))
model.add(LSTM(150))
model.add(Dropout(0.2))
model.add(Dense(12633, activation='softmax'))

I0000 00:00:1786910112.756310     356 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [24]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [25]:
model.fit(X, y, epochs=20, batch_size=256, verbose=1)

Epoch 1/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.0354 - loss: 6.9016
Epoch 2/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.0537 - loss: 6.4843
Epoch 3/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.0800 - loss: 6.2054
Epoch 4/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.0904 - loss: 6.0129
Epoch 5/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.0986 - loss: 5.8558
Epoch 6/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.1042 - loss: 5.7160
Epoch 7/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.1101 - loss: 5.5879
Epoch 8/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.1152 - loss: 5.4690
Epoch 9/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.1195 - loss: 5.3590
Epoch 10/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.1242 - loss: 5.2514
Epoch 11/20
670/670 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.1280 - loss: 5.1485
Epoch 12/20
670/670 ━━━━━━━━━━

In [26]:
model.save('next_word_predictor.h5')